In [7]:
import os
import sys
from datetime import datetime
import itertools

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [8]:
 
# Create one timestamp for the entire experiment batch
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

results_dir = f"results_accuracy_efficiency_{timestamp}"

print(f"Results will be saved to: {results_dir}")


# Define the 9 parameter-matched configurations
mlp_configurations = [
    {"L": 1, "width": 25},
    {"L": 1, "width": 28},
    {"L": 1, "width": 30},
    {"L": 2, "width": 58},
    {"L": 2, "width": 69},
    {"L": 2, "width": 80},
    {"L": 3, "width": 65},
    {"L": 3, "width": 78},
    {"L": 3, "width": 91},
]


# Loop through each matched configuration
for cfg in mlp_configurations:

    layers = cfg["L"]
    width = cfg["width"]

    print(
        f"\n--- Running MLP Experiment: "
        f"Layers (L)={layers}, Width (N)={width} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="MLP",
            hidden_layers=layers,
            hidden_units=width,
            activation=nn.Tanh(),
            adam_lr=1e-3,
            device=device,
            adam_iters=2000,
            lbfgs_iters=2000,
            results_dir=results_dir,
        )

        print(
            f"Success! Time: {compute_time:.2f}s | "
            f"Err U: {err_u:.3e} | Err K: {err_k:.3e}"
        )

    except Exception as e:
        print(
            f"Experiment failed for "
            f"Layers={layers}, Width={width} with error: {e}"
        )

Results will be saved to: results_accuracy_efficiency_2026-09-14_11-02-44

--- Running MLP Experiment: Layers (L)=1, Width (N)=25 ---

[MLP] L=1, N=25 | Params: 1,502 | Mean Err: 5.299e-01 | Saved to 'results_accuracy_efficiency_2026-09-14_11-02-44/'.
Success! Time: 58.05s | Err U: 1.655e-01 | Err K: 8.943e-01

--- Running MLP Experiment: Layers (L)=1, Width (N)=28 ---

[MLP] L=1, N=28 | Params: 1,850 | Mean Err: 3.587e-01 | Saved to 'results_accuracy_efficiency_2026-09-14_11-02-44/'.
Success! Time: 58.80s | Err U: 5.510e-02 | Err K: 6.623e-01

--- Running MLP Experiment: Layers (L)=1, Width (N)=30 ---

[MLP] L=1, N=30 | Params: 2,102 | Mean Err: 3.904e-01 | Saved to 'results_accuracy_efficiency_2026-09-14_11-02-44/'.
Success! Time: 66.63s | Err U: 5.966e-02 | Err K: 7.211e-01

--- Running MLP Experiment: Layers (L)=2, Width (N)=58 ---

[MLP] L=2, N=58 | Params: 14,154 | Mean Err: 5.312e-03 | Saved to 'results_accuracy_efficiency_2026-09-14_11-02-44/'.
Success! Time: 67.81s | Err U: 1.